# 1. Where does a spring chain settle?

Six nodes lie along one axis. Hold the first node at $z_0=1$ and the last at $z_5=3$.
The four interior nodes can move. Our task is to predict where they settle, then
check the prediction with litepgo.

```text
fixed   free   free   free   free   fixed
 z[0] -- z[1] -- z[2] -- z[3] -- z[4] -- z[5]
```

Start with the uneven positions below. Sketch the chain and write down a guess
for its final positions before running the solve.

In [ ]:
#include <cmath>
#include <iostream>
#include <stdexcept>
#include <vector>

#include "litepgo/energy/scalar/quadratic_energy.hpp"
#include "litepgo/linalg/sparse_solver/auto_sparse_solver.hpp"
#include "litepgo/solver/newton.hpp"

static_assert(__cplusplus >= 202002L);
using namespace litepgo;

constexpr int n = 6;
Vec initial = (Vec(n) << 1.0, 1.25, 1.5, 2.0, 2.25, 3.0).finished();
constexpr double tolerance = 1e-10;
std::cout << "Initial positions " << initial.transpose() << '\n';

## Turn five spring differences into one energy

For each spring we use the signed difference $r_i=z_{i+1}-z_i$, with unit
stiffness and zero rest length. The $5\times6$ matrix $D$ has a $-1$ at $(i,i)$
and a $+1$ at $(i,i+1)$. It maps six node positions to five spring differences.
The energy adds up the squares of those differences.

$$
r(z)=Dz,\qquad E(z)=\tfrac12\|Dz\|^2=\tfrac12 z^\top H z,\qquad H=D^\top D.
$$

Give the sparse matrix $H$ to `QuadraticEnergy`. Ask the energy for its value,
six-entry gradient $g$ and $6\times6$ Hessian. The check below compares those
outputs with the formulas $E=\tfrac12 r^\top r$, $g=D^\top r$ and $H=D^\top D$.

In [ ]:
SpMat make_difference_matrix() {
    std::vector<Triplet> entries;
    for (int i = 0; i < n - 1; ++i) {
        entries.emplace_back(i, i, -1.0);
        entries.emplace_back(i, i + 1, 1.0);
    }
    SpMat matrix(n - 1, n);
    matrix.setFromTriplets(entries.begin(), entries.end());
    return matrix;
}

SpMat D = make_difference_matrix();
SpMat H = D.transpose() * D;
energy::QuadraticEnergy chain(H);

Vec stretches = D * initial;
double initial_energy = chain.eval_value(initial);
Vec initial_gradient = chain.eval_gradient(initial);
Eigen::MatrixXd hessian(chain.eval_hessian(initial));

if (!(std::abs(initial_energy - 0.5 * stretches.squaredNorm()) < tolerance &&
      (initial_gradient - D.transpose() * stretches).cwiseAbs().maxCoeff() < tolerance &&
      (hessian - Eigen::MatrixXd(H)).cwiseAbs().maxCoeff() < tolerance)) {
    throw std::runtime_error("Energy derivative check failed");
}

std::cout << "Spring differences " << stretches.transpose() << '\n'
          << "Energy " << initial_energy << '\n'
          << "Gradient " << initial_gradient.transpose() << '\n'
          << "Hessian\n" << hessian << '\n';

## Predict equilibrium from a Hessian row

Pick an interior row of the Hessian. Its three nonzero entries are
$[-1,2,-1]$. Multiplying the row by $z$ gives the gradient at that node.

$$
g_i=-z_{i-1}+2z_i-z_{i+1}
   =(z_i-z_{i-1})-(z_{i+1}-z_i).
$$

At equilibrium each free node has zero gradient, so the two differences in
that row must agree. Applying this to all four free nodes makes all five
spring differences equal. The endpoint difference is $3-1=2$, so the step
is $2/5=0.4$ and the predicted positions are $[1,1.4,1.8,2.2,2.6,3]$.

## Fix the ends and run Newton

`DofReduction` stores the fixed indices $[0,5]$. Newton takes the fixed
**values** from the starting vector and updates only nodes $1$ through $4$.
The solver returns all six positions. Its convergence tolerance applies to
the largest absolute component of the free gradient.

In [ ]:
dof::DofReduction fixed(n, {0, n - 1});
solver::NewtonOptions options{.gradient_tolerance = tolerance};

solver::NewtonResult solve_chain(const Vec& start) {
    linalg::AutoSparseSolver linear_solver;
    return solver::newton_solve(chain, fixed, start, options, linear_solver);
}

auto result = solve_chain(initial);

## Check the positions and read the gradients

Check the status first, then confirm that the endpoints are unchanged,
the positions agree with the equal-spacing prediction, and the free gradient
is below the tolerance. Compare the energy and full gradient before and
after the solve.

In [ ]:
void check_equilibrium(const Vec& start, const solver::NewtonResult& result) {
    if (result.status != solver::NewtonStatus::Converged) {
        throw std::runtime_error("Newton did not converge");
    }
    if (result.x[0] != start[0] || result.x[n - 1] != start[n - 1]) {
        throw std::runtime_error("A fixed endpoint moved");
    }
    const Vec expected = Vec::LinSpaced(n, start[0], start[n - 1]);
    const Vec gradient = chain.eval_gradient(result.x);
    const double position_error = (result.x - expected).cwiseAbs().maxCoeff();
    const double free_gradient = fixed.reduce(gradient).cwiseAbs().maxCoeff();
    if (!(position_error < tolerance && free_gradient < tolerance)) {
        throw std::runtime_error("Equilibrium check failed");
    }
    std::cout << "Status Converged\n"
              << "Positions " << result.x.transpose() << '\n'
              << "Energy " << chain.eval_value(result.x) << '\n'
              << "Gradient " << gradient.transpose() << '\n'
              << "Maximum position error " << position_error << '\n'
              << "Maximum absolute free gradient " << free_gradient << '\n';
}

std::cout << "Before\n"
          << "Energy " << initial_energy << '\n'
          << "Gradient " << initial_gradient.transpose() << "\n\nAfter\n";
check_equilibrium(initial, result);

The energy drops from $0.5$ to $0.4$. The full gradient is approximately
$[-0.4,0,0,0,0,0.4]$. Why can Newton converge with two nonzero entries?
The free nodes have balanced spring forces. Each clamp supplies a force that
balances the spring pulling on its endpoint.

## Try it

Move the right endpoint from $3$ to $4$. Before running the next cell,
write down the new step size and the six positions you expect.
Keep the same four free nodes and rerun the same checks.
What happens to the two endpoint gradients?

In [ ]:
{
    Vec trial = initial;
    trial[n - 1] = 4.0;
    check_equilibrium(trial, solve_chain(trial));
}

<details>
<summary>Check your prediction</summary>

The new step is $(4-1)/5=0.6$. The nodes settle at
$[1,1.6,2.2,2.8,3.4,4]$. The endpoint gradients are approximately $-0.6$
and $+0.6$.

</details>

**Checkpoint** You can read the free-node equilibrium condition from a row of
the energy Hessian and verify it in a constrained solve.

Next, how do five individual spring differences contribute to the gradient at
six nodes? We will follow that map in both directions in Energy 02.